# AE-TFPE Major Revision — Training Campaign

**Run this top to bottom.** It trains models only. It does **not** evaluate on the
test set or on any corruption set — evaluation happens later on your MacBook Pro M4.

## The two method families (never merge them)

| | |
|---|---|
| **Original AE-TFPE** | PE + **ViT-B/16** + **image-space** AE + YOLOv8n-cls. The reference formulation from the submitted manuscript. Reconstructed where the historical implementation was unrecoverable. **Not claimed to be lightweight.** |
| **Efficient AE-TFPE** | PE + **MobileViT-XXS stage 2 (28×28)** + **feature-space** slim denoising AE + YOLOv8n-cls. A **new improvement introduced during the Major Revision** — never presented as the original method. Main candidate: **E5 / C2-28**. |

## Safety properties built into this notebook

- **Google Drive is the source of truth.** Artifacts sync after **every epoch**, so a
  disconnect costs at most one epoch.
- **Resumable.** Re-run the notebook after a disconnect: completed runs are skipped,
  the queue continues.
- **A completed run is never overwritten** without `force=True`.
- **No test access.** `scripts/verify_no_test_access.py` proves at AST level that the
  trainer cannot construct a test or corruption path. Cell 6 runs that proof.

## Preflight and scientific artifacts never share a directory

A smoke run and the scientific run of the same arm have the **same run ID** and write
the **same file names**. The only durable defence is that they are never allowed to
write to the same place:

```
AE_TFPE_MajorRevision/
    preflight/     checkpoints/  logs/  manifest/          SMOKE_TEST = True
    scientific/    checkpoints/  logs/  manifest/  summaries/   SMOKE_TEST = False
```

- `SMOKE_TEST = True` routes **exclusively** to `preflight/`; a preflight campaign
  refuses to run without a per-class limit.
- `SMOKE_TEST = False` routes **exclusively** to `scientific/`; a scientific campaign
  refuses a per-class limit or a smoke flag.
- A scientific run therefore **cannot** adopt a preflight checkpoint — not because a
  check passes, but because the path is not in its namespace.

On top of that, **adoption and resume are provenance-checked, not name-checked**.
Every run stamps `run_provenance.json` with its run ID, namespace, smoke flag, epoch
budget, per-class limits, full-data status, config hash, protocol hash and dataset
hash. `last.pt` carries the same record. A mismatch on **any** field prints a loud
`REFUSED` block and adopts, resumes and overwrites **nothing**.

## Smoke timing is never full-data timing

A preflight epoch trains `4 x 39 = 156` images. A scientific epoch trains **38,584**.
A projection built on the former is not a rough estimate of the latter — it is a
different quantity in the same unit, and it is how a 20-hour campaign gets budgeted at
`0.03 h`.

So under `SMOKE_TEST = True`: measured epoch times are labelled **SMOKE TIMING ONLY**,
are never projected to full data, never update the cost model, and cannot feed the
Cell 15 forced-tier gate. That gate measures **A5 on full data on the actual A100**,
and refuses to run under `SMOKE_TEST` or on any other GPU.

## Two environments — never interchange them

This project runs across two environments with **deliberately different**
dependency stacks. Installing one environment's spec into the other is the
failure mode this section exists to prevent.

### COLAB TRAINING ENVIRONMENT — this notebook

- Uses the **Colab-native, CUDA-compatible `torch` / `torchvision` / `numpy` /
  `Pillow` stack**. These are never downgraded or reinstalled.
- Installs **only training dependencies**, from **`requirements-colab.txt`**
  (via `scripts/colab_setup.sh` in Cell 4).
- **Does not generate corruption datasets.**
- **Does not perform final test evaluation.**

> **Never `pip install -r requirements.txt` on Colab.** That file is the local
> evaluation spec. Its `numpy<2` / `pillow==10.2.0` pins have no wheels for
> current Colab Pythons, so pip source-builds them and the **whole install
> transaction aborts** — leaving `ultralytics` and `timm` missing. The pins are
> also pointless here, because nothing in this notebook touches corruption
> pixels.

### LOCAL EVALUATION ENVIRONMENT — MacBook Pro M4, not this notebook

- Uses the **pinned reproducibility stack** from `requirements.txt`:
  - **Python 3.10.x**
  - **NumPy 1.26.x**
  - **Pillow 10.2.0**
- Used for **corruption generation**, **checksum verification**, and the
  **Normal / Easy / Moderate / Hard evaluation**.
- These pins reproduce `docs/reproducibility_reference.json` exactly. See
  `docs/LOCAL_EVAL_ENVIRONMENT_RECOVERY.md` for how to build that environment.

The split is a **packaging** boundary only. No training or evaluation protocol,
hyperparameter, seed, split, or metric definition differs between them.

## What to do if a cell fails
Every cell below states its failure mode. Nothing is destructive; re-running any cell
is safe.

## CELL 1 — Configuration

The only cell you normally edit. Everything else reads these values.

In [ ]:
# ============================ USER SETTINGS ============================
DRIVE_ROOT   = "/content/drive/MyDrive/AE_TFPE_MajorRevision"   # persistent root
REPO_URL     = "https://github.com/ducthong-dev/VisionTransformer-X-YOLO.git"
REPO_DIR     = "/content/VisionTransformer-X-YOLO"
DATASET_ZIP  = "/content/drive/MyDrive/VisionTransformer_YOLO/dataset/Plant_leaf_diseases_dataset.zip"
DATA_ROOT    = "/content/data/Plant_leaf_diseases_dataset"      # local scratch = fast

# ---------------------------------------------------------------------
# THE ONE SWITCH THAT MATTERS.
#
#   SMOKE_TEST = True   ->  namespace "preflight"   -- plumbing proof, disposable
#   SMOKE_TEST = False  ->  namespace "scientific"  -- the real campaign
#
# The two write to physically separate trees on Drive and cannot see each
# other's checkpoints, logs or manifest:
#
#   AE_TFPE_MajorRevision/preflight/{checkpoints,logs,manifest}
#   AE_TFPE_MajorRevision/scientific/{checkpoints,logs,manifest,summaries}
#
# A scientific run therefore cannot adopt or resume a preflight checkpoint --
# not because a check passes, but because the path is not in its namespace.
# ---------------------------------------------------------------------
SMOKE_TEST            = True    # T4 preflight. Set False only on the A100.
SMOKE_LIMIT_PER_CLASS = 4       # images per class in preflight runs
SMOKE_EPOCHS          = 4       # epochs in preflight runs

# Model-size filter. Models with MORE total parameters than this are SKIPPED by
# default and listed with a reason in Cell 8 -- never silently dropped.
MAX_TRAIN_PARAMS = 20_000_000

# Arms trained despite exceeding the threshold. A3 is included: it is the
# PHYSICAL run behind fusion arm F3, which is satisfied by reuse. F3 is never
# trained separately -- putting it here would train the same model twice.
FORCE_LARGE_IDS = ["A5", "D1", "F1", "F2", "A3", "F4"]

# Campaign wall-clock budget. When the remaining budget cannot fit the next run's
# projection, it is marked SKIPPED_TIME rather than started and lost.
CAMPAIGN_BUDGET_HOURS = 20.0

# Hard gate on the six forced fusion arms (A5, D1, F1, F2, A3, F4). Cell 15 trains
# A5 first ON FULL DATA ON THE A100, measures its REAL epoch time, projects the
# whole forced tier, and STOPS if the projection exceeds this. Smoke timing can
# never feed this gate: Cell 15 refuses to run under SMOKE_TEST.
FORCED_TIER_MAX_HOURS = 12.0

# DataLoader workers. None = keep the FROZEN value (4). Raising it is a PROTOCOL
# AMENDMENT (see Cell 5) and is deliberately NOT taken: the reproducibility
# protocol stays exactly as frozen.
NUM_WORKERS = None

EPOCHS_OVERRIDE = None    # None = the frozen 30. Scientific runs only.
# =======================================================================
import os, time
CAMPAIGN_T0 = time.time()

NAMESPACE = "preflight" if SMOKE_TEST else "scientific"
NS_ROOT   = os.path.join(DRIVE_ROOT, NAMESPACE)
# The five T4 architecture families, one representative arm each. Used by the
# preflight cells and by the readiness gates.
FAMILY_PROBES = {"A": "A0", "E": "E5", "M": "M1", "F": "F1", "B": "B2"}
T4_SMOKE_IDS  = tuple(FAMILY_PROBES.values())

for k, v in dict(DRIVE_ROOT=DRIVE_ROOT, DATA_ROOT=DATA_ROOT).items():
    print(f"{k:16s} {v}")
print(f"{'MAX_PARAMS':16s} {MAX_TRAIN_PARAMS:,}")
print(f"{'FORCE_LARGE':16s} {FORCE_LARGE_IDS or '(none)'}")
print(f"{'BUDGET':16s} {CAMPAIGN_BUDGET_HOURS} h")
print(f"{'SMOKE_TEST':16s} {SMOKE_TEST}")
print(f"{'NAMESPACE':16s} {NAMESPACE}")
print(f"{'ARTIFACT ROOT':16s} {NS_ROOT}")
if SMOKE_TEST:
    print(f"\n  PREFLIGHT MODE: {SMOKE_EPOCHS} epochs, {SMOKE_LIMIT_PER_CLASS} images/class.")
    print("  Everything written this session is plumbing evidence, not science, and")
    print("  every epoch time measured is SMOKE TIMING ONLY.")
else:
    print("\n  SCIENTIFIC MODE: full data, frozen protocol. Nothing here may read the")
    print("  preflight tree.")

## CELL 2 — Mount Google Drive

**Why first:** every artifact is written here. If Drive is not mounted, a disconnect
loses the entire campaign.

**What it creates:** the two isolated namespaces —
`preflight/{checkpoints,logs,manifest}` and
`scientific/{checkpoints,logs,manifest,summaries}`.

**Expected:** `Mounted at /content/drive` and both trees printed. If it reports
**legacy flat directories at the Drive root**, run Cell 2b before anything else.

**On failure:** re-run and complete the authorisation popup. Do not proceed without it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
LAYOUT = {
    "preflight":  ("checkpoints", "logs", "manifest"),
    "scientific": ("checkpoints", "logs", "manifest", "summaries"),
}
for ns, subs in LAYOUT.items():
    for sub in subs:
        os.makedirs(os.path.join(DRIVE_ROOT, ns, sub), exist_ok=True)

print("Drive ready:", DRIVE_ROOT)
for ns, subs in LAYOUT.items():
    print(f"  {ns}/")
    for sub in subs:
        p = os.path.join(DRIVE_ROOT, ns, sub)
        print(f"    {sub:<14} {len(os.listdir(p))} entries")

# The pre-namespace layout wrote everything to <DRIVE_ROOT>/checkpoints and
# <DRIVE_ROOT>/logs, so the T4 smoke runs are sitting exactly where the
# scientific runs are about to go. Cell 2b separates them. Nothing is deleted.
LEGACY = [s for s in ("checkpoints", "logs", "campaign", "completed", "failed")
          if os.path.isdir(os.path.join(DRIVE_ROOT, s))]
if LEGACY:
    print(f"\n  LEGACY FLAT DIRECTORIES FOUND AT THE DRIVE ROOT: {LEGACY}")
    print("  These predate the preflight/scientific split. Run Cell 2b before anything")
    print("  else -- until they are migrated, the isolation gate fails by design.")
else:
    print("\n  no legacy flat directories at the Drive root")

## CELL 2b — Migrate the pre-namespace flat tree (one time)

**Why:** the earlier layout wrote every run to `<DRIVE_ROOT>/checkpoints/<ID>` and
`<DRIVE_ROOT>/logs/<ID>.log` regardless of what it was, so the five T4 architecture
smoke runs (A0, E5, M1, F1, B2 — 4 epochs, 4 images/class) are sitting in exactly the
directories the 30-epoch scientific runs are about to use, and the old manifest
reports them `COMPLETED`.

**What this does:** classifies each legacy run from **its own record** — how many
images it actually trained on versus the split it fingerprinted, and how many epochs
it planned versus the frozen 30 — then **copies** it into the matching namespace and
**moves** the flat tree to `<DRIVE_ROOT>/_legacy_pre_namespace/`.

**Nothing is deleted.** Migrated preflight runs get a stamp that marks them as
plumbing evidence and deliberately omits the identity hashes, so they can never be
adopted or resumed as science.

**Expected:** five runs classified `preflight`, six legacy directories archived.

**On failure:** re-run with `APPLY_MIGRATION = False` and read the dry run.

In [ ]:
# One-time migration of the pre-namespace flat tree. Safe to re-run: it is a
# no-op once the flat directories are gone.
#
# Runs classify themselves from what they recorded -- how many images they
# actually trained on versus the size of the split they fingerprinted, and how
# many epochs they planned versus the config's frozen 30. The five T4 runs
# trained 156 of 38,584 images for 4 of 30 epochs, so they classify as
# preflight. Nothing is deleted: each run is COPIED into its namespace and the
# flat tree is MOVED to <DRIVE_ROOT>/_legacy_pre_namespace/.
import os
print("--- DRY RUN ---")
!python scripts/migrate_campaign_namespaces.py --drive-root "$DRIVE_ROOT"

APPLY_MIGRATION = True     # set False to inspect the dry run only
if APPLY_MIGRATION and any(os.path.isdir(os.path.join(DRIVE_ROOT, s))
                           for s in ("checkpoints", "logs", "campaign")):
    print("\n--- APPLYING ---")
    !python scripts/migrate_campaign_namespaces.py --drive-root "$DRIVE_ROOT" --apply
else:
    print("\nnothing to migrate (or APPLY_MIGRATION is False)")

for ns in ("preflight", "scientific"):
    d = os.path.join(DRIVE_ROOT, ns, "checkpoints")
    print(f"\n{ns}/checkpoints: {sorted(os.listdir(d)) if os.path.isdir(d) else '(none)'}")

## CELL 3 — Repository

**Why:** every artifact records the commit that produced it. A dirty or unknown
commit makes results untraceable.

**Expected:** a commit SHA and `dirty: False`.

**On failure:** delete `REPO_DIR` and re-run.

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
dirty  = bool(subprocess.check_output(["git", "status", "--porcelain"], text=True).strip())
print("commit:", commit)
print("dirty :", dirty)
print(subprocess.check_output(["git", "log", "-1", "--format=%s"], text=True).strip())
assert not dirty, "Working tree is dirty -- artifacts could not be tied to a commit."

# The readiness gate book. Each gate is recorded to
# <DRIVE_ROOT>/preflight/manifest/preflight_gates.json as it is proven, so the
# verdict survives a runtime disconnect instead of living in notebook state.
# Pure stdlib -- importable before any dependency is installed.
from aetfpe import preflight as pf
from aetfpe import provenance as prov
GATES = pf.GateBook(DRIVE_ROOT)
print("\ngate book:", GATES.path)

## CELL 4 — Dependencies

**Why:** installs `requirements-colab.txt`, **not** `requirements.txt`.
`requirements.txt` is the *local evaluation* spec — it pins `numpy<2` and
`pillow==10.2.0` so the corruption pixels match
`docs/reproducibility_reference.json`. Current Colab runtimes publish no wheels
for those versions, so pip source-builds them and the **entire** install
transaction aborts — which is how `ultralytics` and `timm` end up missing.

Those pins are also unnecessary here: this notebook **trains only** and cannot
reach corruption or test data (Cell 6 proves it), so no codec determinism is at
stake. Colab's own CUDA-matched `torch` / `numpy` / `pillow` are left untouched.

**Expected:** every package listed with a version, `cuda avail : True`, and
`np<->torch : ok`.

**On failure:** the script prints the offending package and exits non-zero.
Re-run the pip line without `-q` to see the resolver output. If it reports a
broken `np<->torch` bridge, *Runtime → Restart session* and re-run from Cell 2
(Drive stays mounted).


In [ ]:
!bash scripts/colab_setup.sh

# Assert the property the old `numpy == 1.26.*` check was a proxy for: that the
# packages import and the torch<->numpy bridge actually works on this runtime.
import numpy, torch, ultralytics, timm
print("numpy      ", numpy.__version__)
print("torch      ", torch.__version__)
print("ultralytics", ultralytics.__version__)
print("timm       ", timm.__version__)
bridge_ok = torch.from_numpy(numpy.zeros(4, dtype=numpy.float32)).sum().item() == 0.0
GATES.record("dependency_setup", bridge_ok,
             f"torch {torch.__version__}, numpy {numpy.__version__}, "
             f"ultralytics {ultralytics.__version__}, timm {timm.__version__}")
assert bridge_ok, \
    "torch<->numpy bridge is broken -- Runtime > Restart session, then re-run from Cell 2"

## CELL 5 — GPU verification and the optimization audit

**Why:** this campaign targets an **A100**. Training-quality metrics are
hardware-independent, but **training wall-clock is not** — it is recorded and never
presented as architecture evidence.

**The T4 latency/throughput/memory evidence already collected is separate and is not
reproduced here.** A100 numbers must never be substituted for it.

### Optimization audit — every candidate classified

| Optimization | Class | Applied? |
|---|---|---|
| `pin_memory=True` | **SAFE EXECUTION** — page-locked host buffers; no numeric effect. Also makes the existing `non_blocking=True` transfers actually asynchronous | **yes** |
| `persistent_workers=True` | **SAFE EXECUTION** — avoids re-spawning workers each epoch | **yes** |
| `prefetch_factor` | **SAFE EXECUTION** — queue depth only | **yes** |
| Frozen backbone under `no_grad` | **SAFE** — already in the code | already on |
| `num_workers` 4 → 8 | **PROTOCOL AMENDMENT** — each worker owns an RNG stream, so worker count changes *which* augmentation lands on *which* image. Same distribution, fairness preserved when applied uniformly, but **not bit-reproducible** against earlier runs | **NO** — frozen at 4 |
| **AMP / fp16** | **PROTOCOL AMENDMENT** — fp16 changes numerics, and the frozen protocol sets `amp: false` | **NO** |
| `cudnn.benchmark=True` | **PROTOCOL AMENDMENT** — conflicts with `deterministic=True` | **NO** |
| `torch.compile` | **PROTOCOL AMENDMENT** — kernel substitution can change numerics | **NO** |
| **Frozen-feature caching** | **SCIENTIFIC CHANGE — invalid here.** Training augmentation (RandAugment, flip, erasing) is stochastic per epoch, so cached features would not match the augmented input. It would silently train a different model | **NO** |

Only SAFE EXECUTION optimizations are automatic. **No protocol amendment is taken
in this campaign**: `num_workers` stays at the frozen 4, AMP stays off, cuDNN
benchmarking stays off, `torch.compile` is unused, and no feature caching happens.
The reproducibility protocol is exactly as frozen.

### Crash resume — verified

`train.py` writes `last.pt` every epoch containing **model, optimizer, scheduler,
epoch index, best-so-far, the metric history and all RNG states**. On reconnect the
runner copies it back from Drive into the fresh scratch and training continues from
the next epoch. Tested by killing a run mid-training and resuming from Drive alone:
epochs came back contiguous with the best-so-far preserved.

Before this, the only checkpoint held weights alone — no optimizer moments, no LR
schedule position — so a disconnect at epoch 25 of 30 meant restarting from zero.

In [ ]:
import torch, subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                               "--format=csv,noheader"], text=True).strip())
cuda_ok = torch.cuda.is_available()
assert cuda_ok, "No CUDA device. Runtime -> Change runtime type -> GPU."
GPU = torch.cuda.get_device_name(0)
print("\nGPU        :", GPU)
print("CUDA       :", torch.version.cuda, "| capability", torch.cuda.get_device_capability(0))
print("torch      :", torch.__version__)
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32, "| TF32 cudnn:", torch.backends.cudnn.allow_tf32)
print("cpu cores  :", os.cpu_count())

IS_A100 = "A100" in GPU
GATES.record("cuda", cuda_ok, f"{GPU}, CUDA {torch.version.cuda}, torch {torch.__version__}")
if not IS_A100:
    print(f"\nNOTE: this is a {GPU}, not an A100. Architecture and plumbing preflight is")
    print("      valid on any CUDA device. The forced-tier cost gate (Cell 15) is NOT:")
    print("      it refuses to run anywhere but the A100 it will budget.")
print("\nT4 latency/throughput/memory evidence is SEPARATE and is not re-measured here.")

## CELL 6 — Dataset and the no-test-access proof

**Why:** the dataset is unzipped to **local scratch**, not Drive — reading 55k JPEGs
per epoch over Drive would dominate the runtime.

**Expected:** `38,584 / 8,340 / 8,335` across **39 classes**, and the AST proof exits 0.

**On failure:** a different split means the wrong dataset copy (the sibling ResCBAM
copy has 8,346 / 8,334). Fix the path in Cell 1.

In [ ]:
import os, subprocess, sys
os.environ["DATA_ROOT"]   = DATA_ROOT
os.environ["OUTPUT_ROOT"] = "/content/output"
os.makedirs("/content/data", exist_ok=True); os.makedirs("/content/output", exist_ok=True)

if not os.path.isdir(DATA_ROOT):
    print("unzipping dataset to local scratch ...")
    assert os.path.exists(DATASET_ZIP), f"dataset zip not found: {DATASET_ZIP}"
    subprocess.run(["unzip", "-q", DATASET_ZIP, "-d", "/content/data"], check=True)
else:
    print("dataset already present")

EXPECTED = {"train": 38_584, "val": 8_340, "test": 8_335}
counts = {}
for split in ("train", "val"):
    d = os.path.join(DATA_ROOT, split)
    counts[split] = sum(len(os.listdir(os.path.join(d, c))) for c in os.listdir(d)
                        if os.path.isdir(os.path.join(d, c)))
    print(f"  {split:<6} {counts[split]:>7,} images, {len(os.listdir(d))} classes")
FULL_TRAIN_IMAGES = counts["train"]

counts_ok = all(counts[s] == EXPECTED[s] for s in counts)
GATES.record("dataset_counts", counts_ok,
             f"train {counts['train']:,} / val {counts['val']:,} "
             f"(expected {EXPECTED['train']:,} / {EXPECTED['val']:,})")

!python scripts/verify_dataset.py 2>&1 | tail -6
print("\n--- proof that the trainer cannot reach test or corruption data ---")
rc = subprocess.run([sys.executable, "scripts/verify_no_test_access.py"]).returncode
GATES.record("no_test_access", rc == 0,
             "scripts/verify_no_test_access.py exits 0 (AST proof)" if rc == 0
             else f"scripts/verify_no_test_access.py exited {rc}")

## CELL 7 — Campaign manifests (both namespaces)

**Why:** builds the experiment matrix, measures every model's parameters from the real
code, and merges it into **both** namespace manifests — the preflight one so smoke runs
have somewhere legitimate to go, the scientific one so the readiness gates can inspect
it from this T4 session.

Then two safety steps:

- **`revalidate()`** — the rebuild. Any `COMPLETED` scientific run that cannot prove it
  is a full-data scientific artifact drops back to `PENDING`, with the reason recorded
  on Drive. After this the five T4 smoke runs are **not** counted as completed science.
- **`adopt_existing()`** — provenance-checked. A run on Drive is adopted only if its
  recorded **run ID, config hash, protocol hash, smoke flag, epoch budget, per-class
  limits, full-data status and dataset hash** all match. Any mismatch prints a loud
  `REFUSED` block and adopts nothing.

**Expected:** `scientific COMPLETED: (none)` before the campaign starts.

In [ ]:
import importlib, aetfpe.campaign as cp
importlib.reload(cp)

matrix = cp.build_matrix(max_params=MAX_TRAIN_PARAMS, force_ids=FORCE_LARGE_IDS)

# Both campaigns are constructed every session, whichever mode we are in: the
# readiness gates have to be able to inspect the scientific tree from a preflight
# session, and the preflight tree must exist before the A100 run can prove it was
# kept separate. Constructing them also writes the NAMESPACE marker files.
pre_campaign = cp.Campaign(DRIVE_ROOT, namespace="preflight",
                           scratch_root="/content/campaign_scratch",
                           limit_per_class=SMOKE_LIMIT_PER_CLASS, epochs=SMOKE_EPOCHS,
                           data_root=DATA_ROOT)
sci_campaign = cp.Campaign(DRIVE_ROOT, namespace="scientific",
                           scratch_root="/content/campaign_scratch",
                           epochs=EPOCHS_OVERRIDE, data_root=DATA_ROOT)
pre_campaign.seed(matrix, gpu=GPU)
sci_campaign.seed(matrix, gpu=GPU)

# Rebuild step: any COMPLETED scientific run that cannot prove it is a full-data
# scientific artifact drops back to PENDING, with the reason recorded on Drive.
rebuild = sci_campaign.revalidate()
if rebuild["demoted"]:
    print("SCIENTIFIC MANIFEST REBUILD -- demoted:")
    for d in rebuild["demoted"]:
        print(f"  {d['id']}: {'; '.join(d['why'])}")
else:
    print("scientific manifest rebuild: nothing to demote")

# Adoption is provenance-checked. A pre-existing run is only adopted if its
# recorded run ID, config hash, protocol hash, smoke flag, epoch budget,
# per-class limits, full-data status and dataset hash all match this campaign.
for rid in [e["id"] for e in cp.EXPERIMENTS]:
    sci_campaign.adopt_existing(rid)

campaign = pre_campaign if SMOKE_TEST else sci_campaign
print(f"\nACTIVE NAMESPACE : {campaign.namespace}")
print(f"checkpoints      : {campaign.ckpt_dir}")
print(f"logs             : {campaign.log_dir}")
print(f"manifest         : {campaign.manifest_path}")
print(f"summary          : {campaign.summary_path}")

print("\nstatus counts:", campaign.counts())
print("P0 queue:", campaign.queue("P0"))
print("P1 queue:", campaign.queue("P1"))
print("P2 queue:", campaign.queue("P2"))
print("\nscientific COMPLETED:", sci_campaign.completed_ids() or "(none -- as expected before the campaign)")

## CELL 8 — Included / skipped models, with reasons

**Nothing is silently skipped.** Six over-threshold arms (`A5 D1 F1 F2 A3 F4`) are
already forced in Cell 1; the second table should list only `A2`, `B1` and `B3`.

In [ ]:
inc = [r for r in matrix if r["status"] == cp.PENDING]
skp = [r for r in matrix if r["status"] == cp.SKIPPED_SIZE]

print("=" * 118); print("INCLUDED"); print("=" * 118)
print(f"{'ID':<4}{'pri':<5}{'grp':<4}{'model':<34}{'params':>12}{'trainable':>12}  reviewer question")
for r in sorted(inc, key=lambda x: (x["priority"], x["id"])):
    print(f"{r['id']:<4}{r['priority']:<5}{r['group']:<4}{r['title'][:33]:<34}"
          f"{r['params']:>12,}{r['trainable_params']:>12,}  {r['reviewer']}")

print("\n" + "=" * 118); print(f"SKIPPED -- over MAX_TRAIN_PARAMS = {MAX_TRAIN_PARAMS:,}"); print("=" * 118)
print(f"{'ID':<4}{'grp':<4}{'model':<30}{'total':>12}{'trainable':>12}{'frozen':>12}  reviewer question")
for r in sorted(skp, key=lambda x: x["id"]):
    print(f"{r['id']:<4}{r['group']:<4}{r['title'][:29]:<30}{r['params']:>12,}"
          f"{r['trainable_params']:>12,}{r['params']-r['trainable_params']:>12,}  {r['reviewer']}")

print("""
------------------------------------------------------------------------------
READ THIS BEFORE ACCEPTING THE SKIP LIST
------------------------------------------------------------------------------
Seven of the nine arms over the threshold (A2 A3 A5 D1 F1 F2 F4) carry only
~1.5-1.75M TRAINABLE parameters. The other ~85.8M is a FROZEN ViT-B/16 that runs under
no_grad -- so the cost is its FORWARD pass, not optimiser work. A total-parameter
threshold therefore over-states how expensive they are to train, while still
correctly flagging them as slow: measured on a T4 they run at 92.8 img/s versus
the baseline's 4,365 img/s (47x).

SIX OF THEM ARE FORCED THIS CAMPAIGN: A5 D1 F1 F2 A3 F4. Only A2, B1 and B3
remain skipped, so the table above should list exactly those three.

What the remaining skip costs, per group:
  A2       TF-only, Original-side                     -> Efficient-side E3/E5 partly covers this
  B1, B3   ResNet-50 / ViT-B/16 external baselines    -> B2 (EfficientNet-B0) covers
           these are FULLY trainable: genuinely expensive  the external-baseline role

Why A3 is forced: it is the PHYSICAL run behind fusion arm F3. F3 has an identical
config signature (confirmed by scripts/print_run_matrix.py) and is satisfied by
REUSE of A3's checkpoint -- it is never trained separately. Forcing A3 therefore
buys the Original-side no-AE control AND the F3 fusion arm for one training.

Can the skipped arms be represented without training?
  * Complexity / efficiency claims  -> YES. Already measured on a T4 for all five
    architectures and archived in docs/evidence/. Training changes none of it.
  * Original AE-TFPE as a reference -> YES, as a computational/reference formulation
    (see docs/EXPERIMENT_CAMPAIGN_V2_PLAN.md).
  * Fusion superiority (F1/F2/F4 vs D1) -> NO. This is an accuracy comparison and
    cannot be made from architecture analysis. Skipping it means the manuscript
    cannot claim AE fusion is superior; that claim must be withdrawn or deferred.
  * Component ablation on the Original side -> PARTIALLY, via the Efficient side,
    but conclusions are NOT assumed to transfer (different encoder AND AE space).

The AE-fusion superiority claim and the denoising-objective claim need
A5 + D1 + F1 + F2 + F4, and F3 needs A3: six forced physical trainings, already set
in FORCE_LARGE_IDS in Cell 1. Cell 9 projects what they cost; Cell 15 gates them on a
measured FULL-DATA epoch time from the A100 itself.
------------------------------------------------------------------------------""")

## CELL 9 — Compute budget projection

**These are estimates until measured.** After the first epoch of each architecture
family the runner prints the *measured* epoch time and re-projects — that is the number
to trust.

Basis: T4 batch-32 throughput (measured), training taken at ~⅓ of inference throughput,
46,924 images per epoch, then adjusted for the detected GPU.

In [ ]:
N_IMG   = 46_924
EPOCHS  = EPOCHS_OVERRIDE or 30
T4_IPS  = {"small": 4365.6, "mobilevit": 774.8, "vit": 92.8}   # measured, batch 32
SPEEDUP = 3.0 if IS_A100 else 1.0                              # rough A100:T4 for fp32
LOADER_FLOOR_S = 75.0     # see note below

def family(r):
    if r["params"] > 20_000_000:                        return "vit"
    if "mobilevit" in r["config"] or r["group"] == "E": return "mobilevit"
    return "small"

def train_multiplier(r):
    """Training cost / inference cost.

    Backward runs only over TRAINABLE parameters. The ViT-B/16 arms keep 96.7% of
    their forward FLOPs in a frozen branch executed under no_grad, so a flat
    'training = 3x inference' rule overstates them by ~2.9x. Measured split for
    A5: 33.7266 frozen + 1.1465 trainable of 34.8731 GFLOPs -> 1.066x, not 3x.
    """
    share = r["trainable_params"] / max(r["params"], 1)
    return 1.0 + 2.0 * share

def project(r):
    """A-PRIORI estimate from the T4 batch-32 throughput benchmark.

    Its inputs are the standalone throughput measurements and the full 46,924
    images per epoch. It never reads a run's measured epoch time, so a smoke run
    cannot leak into it -- see `record_full_data_epoch_time` in Cell 10 for the
    one place measured timing is allowed to update anything.
    """
    ips = T4_IPS[family(r)] / train_multiplier(r) * SPEEDUP
    compute_s = N_IMG / ips
    # Small models are usually data-loader bound, not compute bound: 46,924 JPEG
    # decodes + RandAugment per epoch on 4 workers. Take whichever dominates.
    return max(compute_s, LOADER_FLOOR_S) * EPOCHS / 3600.0

if SMOKE_TEST:
    print("=" * 78)
    print("SMOKE_TEST=True -- the campaign budget below is NOT the workload of this")
    print("session. A preflight epoch trains SMOKE_LIMIT_PER_CLASS x 39 images; the")
    print("scientific epoch trains 38,584. The projections here describe the")
    print("SCIENTIFIC campaign and are shown for reference only. No number measured")
    print("in this session feeds them.")
    print("=" * 78 + "\n")

print(f"GPU {GPU} | epochs {EPOCHS} | {N_IMG:,} images/epoch | assumed A100:T4 speedup {SPEEDUP}x")
print(f"data-loader floor assumed {LOADER_FLOOR_S:.0f} s/epoch at num_workers=4\n")
tot = {}
for pri in ("P0", "P1", "P2"):
    rows = [r for r in matrix if r["priority"] == pri and r["status"] == cp.PENDING]
    h = sum(project(r) for r in rows)
    tot[pri] = h
    print(f"{pri}: {len(rows)} runs -> ~{h:.1f} h   " + ", ".join(r["id"] for r in rows))
print(f"\nP0+P1 projected: ~{tot['P0']+tot['P1']:.1f} h   (budget {CAMPAIGN_BUDGET_HOURS} h)")

big = [r for r in matrix if r["status"] == cp.SKIPPED_SIZE]
print(f"\nIf you forced the skipped arms instead:")
for r in sorted(big, key=lambda x: x["id"]):
    print(f"   {r['id']:<4}{r['title'][:34]:<36} ~{project(r):>5.1f} h")
print(f"   {'ALL':<4}{'(not recommended in one day)':<36} ~{sum(project(r) for r in big):>5.1f} h")
forced_rows = [r for r in matrix if r["id"] in cp.FORCED_FUSION_IDS]
print(f"   {'FORCED ' + '+'.join(cp.FORCED_FUSION_IDS):<40} ~{sum(project(r) for r in forced_rows):>5.1f} h")
print("""
ESTIMATES ONLY -- the numbers to trust are the MEASURED FULL-DATA epoch times the
runner prints. Two things drive the estimates above:
  * backward runs only over TRAINABLE parameters, so the ViT-B/16 arms cost about
    1.07x their inference cost, not 3x (96.7% of their forward is frozen);
  * small models are data-loader bound, so P0 runs take a similar wall-clock
    regardless of parameter count.
The forced tier is additionally gated on its own measured FULL-DATA epoch time on
the A100 in Cell 15.""")

## CELL 10 — Training runner

**Why:** one function so every arm gets identical treatment. It prints the frozen
protocol, applies only SAFE execution optimizations, syncs to Drive after every epoch,
and re-projects the remaining campaign from the *measured* first-epoch time.

**On failure:** the run is marked `FAILED` with its log path; the campaign continues to
the next arm. Re-running the cell retries failed runs.

In [ ]:
import csv, json, time

# The scientific cost model. Only FULL-DATA measurements are allowed in.
COST_MODEL = {}

def record_full_data_epoch_time(rid):
    """Take a measured epoch time into the cost model, or refuse it loudly.

    A preflight epoch trains 156 images; the scientific epoch trains 38,584 --
    0.4%. A projection built on the former is not a rough estimate of the
    latter, it is a different quantity in the same unit, and it is how a 20-hour
    campaign gets budgeted at 0.03 h. `assert_projectable` reads the timing basis
    recorded in the run's own provenance stamp and raises unless it is FULL_DATA.
    """
    try:
        ep_s = campaign.assert_projectable(rid)
    except prov.ProvenanceMismatch as exc:
        print(exc)
        return None
    COST_MODEL[rid] = ep_s
    return ep_s

def protocol_banner():
    from aetfpe.config import load_experiment, build_protocol
    c = load_experiment("configs/baseline_rgb.yaml"); p = build_protocol(c)
    tr = os.path.join(DATA_ROOT, "train"); va = os.path.join(DATA_ROOT, "val")
    ntr = sum(len(os.listdir(os.path.join(tr,x))) for x in os.listdir(tr) if os.path.isdir(os.path.join(tr,x)))
    nva = sum(len(os.listdir(os.path.join(va,x))) for x in os.listdir(va) if os.path.isdir(os.path.join(va,x)))
    print("=" * 78); print("FROZEN PROTOCOL -- identical for every comparable arm"); print("=" * 78)
    for k, v in [("namespace", NAMESPACE), ("dataset", DATA_ROOT),
                 ("train images", f"{ntr:,}"), ("val images", f"{nva:,}"),
                 ("classes", len(os.listdir(tr))), ("image size", p.img_size),
                 ("epochs", SMOKE_EPOCHS if SMOKE_TEST else (EPOCHS_OVERRIDE or p.epochs)),
                 ("batch size", p.batch_size),
                 ("optimizer", p.optimizer), ("lr", p.lr), ("weight decay", p.weight_decay),
                 ("scheduler", "cosine + 3-epoch warm-up"), ("seed", f"{p.seed} (deterministic={p.deterministic})"),
                 ("augmentation", "hflip 0.5, RandAugment(2,9), RandomErasing 0.4"),
                 ("checkpoint", p.checkpoint_selection), ("AMP", f"{p.amp}  (frozen off)"),
                 ("pretrained", "True (ImageNet / COCO transfer)"),
                 ("AE warm-up", f"{p.ae_warmup_epochs} reconstruction-only epochs (AE arms)"),
                 ("num_workers", f"{NUM_WORKERS or p.num_workers}"),
                 ("TEST EVALUATION", "NONE -- validation only, done later on the Mac")]:
        print(f"  {k:<16} {v}")
    if SMOKE_TEST:
        print(f"  {'IMAGES/CLASS':<16} {SMOKE_LIMIT_PER_CLASS}   <<< PREFLIGHT SUBSET >>>")
    print("=" * 78)

def train_one(rid, force=False):
    extra = []
    if NUM_WORKERS:  extra += ["--num-workers", str(NUM_WORKERS)]
    # Epochs and the per-class limit belong to the campaign object, not to this
    # function: they are part of the run's identity and must match what the
    # provenance check expects.
    if not SMOKE_TEST:
        used_h = (time.time() - CAMPAIGN_T0) / 3600.0
        left_h = CAMPAIGN_BUDGET_HOURS - used_h
        row = [r for r in matrix if r["id"] == rid]
        proj = project(row[0]) if row else 0.0
        if proj > left_h:
            campaign.manifest["runs"][rid].update(
                status=cp.SKIPPED_TIME,
                reason=f"projected {proj:.1f} h exceeds the {left_h:.1f} h remaining")
            campaign.save()
            print(f"[{rid}] SKIPPED_TIME -- projected {proj:.1f} h > {left_h:.1f} h remaining")
            return campaign.manifest["runs"][rid]

    r = campaign.run(rid, extra_args=extra, force=force, gpu=GPU)

    m = os.path.join(campaign.ckpt_dir, rid, "metrics.csv")
    if os.path.exists(m):
        rows = list(csv.DictReader(open(m)))
        if rows:
            e1 = float(rows[0]["seconds"])
            if SMOKE_TEST:
                n_used = int(float(rows[0].get("train_n") or 0))
                print(f"[{rid}] SMOKE TIMING ONLY: epoch 1 = {e1:.1f}s on {n_used:,} train "
                      f"images ({n_used / max(FULL_TRAIN_IMAGES, 1):.2%} of the "
                      f"{FULL_TRAIN_IMAGES:,}-image scientific epoch).")
                print(f"[{rid}] NOT projected to full data, NOT added to the cost model, "
                      "NOT usable for the P2/A100 gate.")
            else:
                n = EPOCHS_OVERRIDE or 30
                print(f"[{rid}] MEASURED FULL-DATA epoch 1: {e1:.1f}s -> projected "
                      f"{e1*n/3600:.2f} h for {n} epochs")
                record_full_data_epoch_time(rid)
    print(f"[{rid}] campaign elapsed {(time.time()-CAMPAIGN_T0)/3600:.2f} h "
          f"of {CAMPAIGN_BUDGET_HOURS} h\n")
    return r

def run_tier(pri):
    assert not SMOKE_TEST, ("run_tier() trains a whole priority tier and is for the "
                            "scientific campaign. Under SMOKE_TEST use the preflight "
                            "cells 10a-10c.")
    q = campaign.queue(pri)
    print(f"\n########## {pri}: {len(q)} run(s) -> {q}\n")
    for rid in q:
        train_one(rid)
    print(f"########## {pri} done. counts: {campaign.counts()}")

def show_summary():
    import pandas as pd
    df = pd.read_csv(campaign.summary_path)
    cols = [c for c in ["id","model","priority","status","namespace","smoke_test",
                        "timing_basis","params","best_val_top1","best_val_top5",
                        "runtime_s","gpu"] if c in df.columns]
    display(df[cols])
    done = df[df.status == "COMPLETED"]
    print(f"[{campaign.namespace}] COMPLETED {len(done)} | "
          f"elapsed {(time.time()-CAMPAIGN_T0)/3600:.2f} h")
    p0 = df[(df.priority == "P0")]
    ok = len(p0[p0.status == "COMPLETED"])
    print(f"P0: {ok}/{len(p0)} complete " +
          ("<<< SCIENTIFIC MINIMUM COMPLETE >>>" if ok == len(p0) and len(p0) and not SMOKE_TEST
           else "(incomplete)" if not SMOKE_TEST else "(preflight namespace -- not science)"))

protocol_banner()
print(f"\nrunner ready [{NAMESPACE}]: train_one(id) | run_tier('P0') | show_summary()")

## CELL 10a — PREFLIGHT: architecture smoke, all five families

**Requires `SMOKE_TEST = True`.** Everything written here lands in `preflight/`.

One representative arm per architecture family — `A0` (YOLOv8n-cls), `E5`
(MobileViT), `M1` (legacy LUT), `F1` (fusion), `B2` (EfficientNet) — proves the model
builds, the CUDA path runs, and artifacts reach Drive. A family already evidenced in
`preflight/checkpoints/` (including the migrated T4 runs) is **not** re-run.

**Epoch times printed here are SMOKE TIMING ONLY.** They are labelled as such, are not
projected to full data, do not enter the cost model, and cannot feed the Cell 15 gate.

In [ ]:
assert SMOKE_TEST, "Set SMOKE_TEST = True in Cell 1. This cell writes preflight artifacts."
assert campaign.namespace == "preflight", campaign.namespace

# One representative arm per architecture family. Families already evidenced in
# the preflight tree -- including the five migrated T4 runs -- are not repeated.
have = pf.completed_preflight_runs(DRIVE_ROOT)
print("preflight evidence already on Drive:", sorted(have) or "(none)")

for letter, rid in FAMILY_PROBES.items():
    hits = [r for r in have if r[:1] == letter]
    if hits:
        print(f"\nfamily {letter}: satisfied by {hits} -- not re-running")
        continue
    print(f"\nfamily {letter}: no evidence, running {rid}")
    train_one(rid)

print("\n" + "=" * 78)
smoke_gates = pf.evaluate_architecture_smoke(DRIVE_ROOT)
GATES.record_many(smoke_gates)
GATES.record("drive_persistence", *pf.evaluate_drive_persistence(DRIVE_ROOT, min_runs=5))
print("=" * 78)

## CELL 10b — PREFLIGHT: Drive-only resume

**Requires `SMOKE_TEST = True`.** The claim the whole one-day campaign rests on is
that a Colab disconnect costs one epoch, not the run. This tests it destructively:

1. train a disposable `RESUME_PROBE` far enough to write `last.pt`, mirrored to Drive
2. **delete the `/content` scratch directory outright**
3. restore from **Drive and nothing else**
4. resume, and verify it starts at the **next** epoch, that `metrics.csv` is contiguous
   `1..N` with the pre-crash rows unchanged, and that best-so-far survived
5. **negative control:** offer the same directory to a *different* experiment
   (a different epoch budget) and verify it is **refused**

No scientific artifact is read or written; the script refuses to run outside
`preflight/`. Evidence lands in `preflight/manifest/resume_test.json`.

In [ ]:
# Drive-only resume preflight. Entirely inside the preflight namespace: it
# trains a disposable RESUME_PROBE run, deletes /content scratch outright,
# restores from Drive alone, resumes, and checks that it continues at the next
# epoch with contiguous metrics and the best-so-far intact. Step 5 is a negative
# control: the same directory is offered to a DIFFERENT experiment and must be
# refused.
import subprocess, sys
rc = subprocess.run(
    [sys.executable, "scripts/preflight_resume_test.py",
     "--drive-root", DRIVE_ROOT,
     "--scratch-root", "/content/campaign_scratch/preflight_resume",
     "--device", "cuda",
     "--limit-per-class", "2", "--epochs", "4", "--stop-after", "2"]).returncode

ok, detail = pf.evaluate_resume(DRIVE_ROOT)
GATES.record("drive_only_resume", ok and rc == 0, detail)
print(f"\nevidence: {os.path.join(DRIVE_ROOT, 'preflight', 'manifest', 'resume_test.json')}")

## CELL 10c — FINAL READINESS

Re-evaluates every derivable gate **from the artifacts on Drive**, not from notebook
variables, so a gate cannot pass because an earlier cell was run and then edited.

Prints one line per gate and then exactly one of:

```
READY FOR A100 FULL CAMPAIGN
NOT READY FOR A100 FULL CAMPAIGN
```

with the failed gates listed underneath. Do not switch to the A100 until this prints
the first sentence.

In [ ]:
# ---------------------------------------------------------------------------
# FINAL READINESS. Re-evaluates every derivable gate from the artifacts on Drive
# -- not from notebook variables -- then prints one of exactly two sentences.
# ---------------------------------------------------------------------------
import importlib
importlib.reload(pf)

GATES.record_many(pf.evaluate_architecture_smoke(DRIVE_ROOT))
GATES.record("drive_persistence", *pf.evaluate_drive_persistence(DRIVE_ROOT, min_runs=5))
GATES.record("drive_only_resume", *pf.evaluate_resume(DRIVE_ROOT))
GATES.record("isolation", *pf.evaluate_isolation(DRIVE_ROOT))
GATES.record("scientific_manifest_clean",
             *pf.evaluate_scientific_manifest_clean(DRIVE_ROOT, T4_SMOKE_IDS))
GATES.record("a3_forced", *pf.evaluate_a3_forced(FORCE_LARGE_IDS, sci_campaign))

print()
READY_FOR_A100 = GATES.report()

## CELL 11 — P0: the scientific minimum

**These six determine whether the revised paper is defensible.**

`A0` fair baseline · `E5` Efficient AE-TFPE · `M1/M2/M3` mechanism controls · `E3` AE-removed control.

**The mechanism gate:** if E5 does not clearly beat the best of M1/M2/M3, the
contribution is the input transform, not the architecture — and that is the finding.
Stop and report it rather than spending the rest of the budget.

In [ ]:
run_tier('P0')

## CELL 12 — Checkpoint after P0

In [ ]:
show_summary()

## CELL 13 — P1: high value

`A1` PE-only (serves **both** method families — with no TF branch the Original and
Efficient variants are the same model) · `A4` RGB+AE · `E7` C2-7 spatial control ·
`B2` EfficientNet-B0 external baseline.

**Do not skip to Cell 15 before this finishes.** The forced P2 arms are the
expensive ones; P1 is cheap and answers component questions.

In [ ]:
run_tier('P1')

## CELL 14 — Checkpoint after P1

In [ ]:
show_summary()

## CELL 15 — P2: the forced fusion tier

**Forced this campaign: `A5, D1, F1, F2, A3, F4`** — the six physical trainings that
carry the AE-fusion superiority claim and the denoising-objective claim (A5 − D1).
Without them those claims must be withdrawn or deferred.

`A3` is in the set because it is the **physical run behind fusion arm F3**. `F3` is a
logical reuse of `A3`'s checkpoint (identical config signature, confirmed by
`scripts/print_run_matrix.py`) and is **never trained separately** — doing so would
train the same model twice and report it as two experiments.

`B1` (ResNet-50) and `B3` (ViT-B/16) stay skipped: both are **fully trainable** and
therefore genuinely expensive, and `B2` (EfficientNet-B0) already fills the external
-baseline role.

**The gate refuses to run under `SMOKE_TEST`, and refuses to run off an A100.** It
budgets six multi-hour arms from one measurement, so that measurement must be full
data on the GPU that will run them.

Each forced arm carries the frozen ViT-B/16, so `last.pt` is ~365 MB and is written
to Drive every epoch. Budget roughly **11 GB of Drive writes per arm** and ~4 GB of
resident Drive space across the six.

Runs whose projection does not fit the remaining budget are marked `SKIPPED_TIME`
rather than started and lost to a timeout.

In [ ]:
# The forced-tier cost gate. It measures ONE arm and budgets the rest from it, so
# the measurement has to be the real thing: full data, on the GPU that will run
# the tier. Both conditions are enforced rather than assumed.
if SMOKE_TEST:
    raise SystemExit("REFUSED: the forced-tier gate cannot run under SMOKE_TEST. A smoke "
                     "epoch trains a per-class subset; budgeting 6 multi-hour arms from it "
                     "is meaningless. Set SMOKE_TEST = False and re-run from Cell 1.")
if not IS_A100:
    raise SystemExit(f"REFUSED: this is a {GPU}. The forced tier must be budgeted from a "
                     "FULL-DATA measurement on the A100 that will actually run it. "
                     "Switch the runtime to an A100 and re-run from Cell 1.")

q = campaign.queue("P2")
if not q:
    print("P2 queue empty -- nothing forced. Set FORCE_LARGE_IDS in Cell 1 to add arms.")
else:
    # GATE: train ONE forced arm first, measure it, then decide with real numbers
    # instead of an estimate. A5 is the representative -- D1/F1/F2/A3/F4 share its
    # frozen ViT-B/16 branch and are within a few percent of its cost.
    probe = "A5" if "A5" in q else q[0]
    print(f"GATE: training {probe} first on FULL DATA to measure the real cost of this tier.\n")
    train_one(probe)

    ep_s = record_full_data_epoch_time(probe)
    if ep_s is None:
        print(f"No usable FULL-DATA timing for {probe}; not proceeding blindly. "
              f"Inspect {campaign.namespace}/logs/{probe}.log.")
    else:
        per_run_h = ep_s * (EPOCHS_OVERRIDE or 30) / 3600.0
        remaining = [r for r in campaign.queue("P2")]
        tier_h = per_run_h * len(remaining)
        print(f"\nMEASURED {probe} on FULL DATA: {ep_s:.0f} s/epoch -> {per_run_h:.2f} h per forced arm")
        print(f"{len(remaining)} arm(s) left ({remaining}) -> {tier_h:.2f} h")
        print(f"gate: FORCED_TIER_MAX_HOURS = {FORCED_TIER_MAX_HOURS} h")

        if tier_h > FORCED_TIER_MAX_HOURS:
            print(f"""
STOP -- the forced tier projects {tier_h:.1f} h, over the {FORCED_TIER_MAX_HOURS} h gate.
Not started. {probe} is COMPLETED and safe on Drive.

Your options, in order of scientific value:
  1. Run D1 only  (~{per_run_h:.1f} h). With A5 already done, A5-D1 isolates the
     DENOISING OBJECTIVE -- the specific claim Reviewer #10.4 challenges.
     This is the highest-value single arm remaining.
  2. Add A3 (~{per_run_h:.1f} h more). A3 is the Original-side no-AE control AND
     the physical run behind fusion arm F3, so it answers two questions at once.
  3. Add F1 and F2 (~{2*per_run_h:.1f} h more) for a partial fusion comparison,
     disclosed as partial.
  4. Raise FORCED_TIER_MAX_HOURS and re-run this cell if you have the wall-clock.

To run a specific arm:  train_one("D1")""")
        else:
            print(f"\nGATE PASSED -- {tier_h:.1f} h fits the {FORCED_TIER_MAX_HOURS} h budget. Continuing.\n")
            run_tier("P2")

## CELL 16 — Final campaign summary

In [ ]:
show_summary()
import pandas as pd
df = pd.read_csv(campaign.summary_path)
print("\nby status:"); print(df.status.value_counts().to_string())
print(f"\nnamespace: {campaign.namespace}")
print(f"manifest : {campaign.manifest_path}")
print(f"summary  : {campaign.summary_path}")
print(f"checkpts : {campaign.ckpt_dir}")
if SMOKE_TEST:
    print("\nPREFLIGHT NAMESPACE -- none of the above is a scientific result.")
else:
    print("\nNo test evaluation was run. No corruption set was generated.")

## CELL 17 — Export for local evaluation on the MacBook Pro M4

Copies only what evaluation needs. Full details in `docs/LOCAL_EVALUATION_HANDOFF.md`.

**Each completed run's folder contains:** `checkpoint.pt` (best val top-1, with its
config and class list embedded), `config.yaml`, `metrics.csv`, `train_summary.json`,
`environment.json`.

In [ ]:
import tarfile, os, json
assert not SMOKE_TEST and campaign.namespace == "scientific", (
    "REFUSED: the local-evaluation export ships scientific checkpoints only. "
    f"Active namespace is {campaign.namespace!r}.")

out = os.path.join(DRIVE_ROOT, "scientific", "summaries", "for_local_evaluation.tar.gz")
os.makedirs(os.path.dirname(out), exist_ok=True)
done = sci_campaign.completed_ids()

# Every exported run must still carry a valid full-data scientific stamp at the
# moment of export, so a contaminated artifact cannot reach the evaluation host.
bad = [rid for rid in done
       if prov.is_smoke(prov.load(os.path.join(sci_campaign.ckpt_dir, rid)))]
assert not bad, f"REFUSED: smoke-stamped artifacts in the scientific tree: {bad}"

with tarfile.open(out, "w:gz") as tar:
    for rid in done:
        d = os.path.join(sci_campaign.ckpt_dir, rid)
        if os.path.isdir(d):
            tar.add(d, arcname=rid)
    tar.add(sci_campaign.summary_path, arcname="campaign_summary.csv")
    tar.add(sci_campaign.manifest_path, arcname="campaign_manifest.json")
print(f"wrote {out}  ({os.path.getsize(out)/1e6:.1f} MB)  runs: {done}")
print(f"""
NEXT, ON YOUR MAC:
  1. Download {out} from Drive
  2. mkdir -p results/campaign && tar -xzf for_local_evaluation.tar.gz -C results/campaign
  3. Follow docs/LOCAL_EVALUATION_HANDOFF.md for Normal / Easy / Moderate / Hard

Evaluation was deliberately NOT run here.""")